# Kimi — бесплатный ИИ-ассистент

Запускайте ячейки по порядку. Для лучшей скорости включите GPU в **Среда выполнения → Сменить тип среды выполнения → T4 GPU** (если доступно).

In [ ]:
%pip -q install -U "transformers>=4.46.0" "accelerate>=1.0.0" "gradio>=5.0.0" huggingface_hub

In [ ]:
import os, json, time, torch
from pathlib import Path
from datetime import datetime, timezone
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio as gr

# Основная модель с минимальными отказами. Если не загружается, замените на Qwen/Qwen2.5-3B-Instruct.
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")
MAX_NEW_TOKENS = 768
MAX_CONTEXT_MESSAGES = 16
SYSTEM_PROMPT = "Ты — полезный ИИ-ассистент. Отвечай на языке пользователя, обычно по-русски. Будь точным; если не уверен, прямо скажи об этом."

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
else: print("GPU не обнаружен. Модель может быть очень медленной или не поместиться в RAM; бесплатный GPU не гарантируется.")

## Подключение Google Drive

Это нужно только для сохранения истории между сессиями. История хранится в `MyDrive/KimiAssistant/history.json`.

In [ ]:
USE_GOOGLE_DRIVE = True
HISTORY_DIR = Path("/content/KimiAssistant")
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        HISTORY_DIR = Path('/content/drive/MyDrive/KimiAssistant')
    except Exception as e:
        print('Не удалось подключить Google Drive; история будет только в текущей сессии:', e)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_FILE = HISTORY_DIR / 'history.json'
def load_saved_history():
    try:
        data = json.loads(HISTORY_FILE.read_text(encoding='utf-8'))
        return data if isinstance(data, list) else []
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        return []
def save_history(messages):
    tmp = HISTORY_FILE.with_suffix('.tmp')
    tmp.write_text(json.dumps(messages, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(HISTORY_FILE)
saved_messages = load_saved_history()
print(f'Сохранённых сообщений: {len(saved_messages)}; файл: {HISTORY_FILE}')

In [ ]:
print('Загружаю модель:', MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
load_kwargs = {"device_map": "auto"}
if torch.cuda.is_available():
    load_kwargs["torch_dtype"] = torch.float16
else:
    load_kwargs["torch_dtype"] = torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print('Модель загружена.')

In [ ]:
def generate_reply(message, history):
    history = history or []
    recent = history[-MAX_CONTEXT_MESSAGES:]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for item in recent:
        if isinstance(item, dict) and item.get('role') in ('user', 'assistant') and isinstance(item.get('content'), str):
            messages.append({"role": item['role'], "content": item['content']})
    messages.append({"role": "user", "content": message})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.08, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer or 'Не удалось получить ответ. Попробуйте ещё раз.'

def respond(message, chat_history):
    if not message or not message.strip():
        return chat_history, ""
    chat_history = list(chat_history or [])
    answer = generate_reply(message.strip(), chat_history)
    now = datetime.now(timezone.utc).isoformat()
    chat_history.extend([
        {"role": "user", "content": message.strip(), "time": now},
        {"role": "assistant", "content": answer, "time": now}
    ])
    try:
        save_history(chat_history)
    except Exception as e:
        print('Не удалось сохранить историю:', e)
    return chat_history, ""

def clear_chat():
    try:
        save_history([])
    except Exception as e:
        print('Не удалось очистить сохранённую историю:', e)
    return [], [], None

with gr.Blocks(title='Kimi — ИИ-ассистент', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Kimi — ИИ-ассистент\nБесплатный запуск модели в текущей Colab-сессии. Не отправляйте секретные данные.')
    chatbot = gr.Chatbot(value=[m for m in saved_messages if m.get('role') in ('user','assistant')], height=520, label='Переписка')
    state = gr.State([m for m in saved_messages if m.get('role') in ('user','assistant')])
    with gr.Row():
        message = gr.Textbox(placeholder='Напишите сообщение…', show_label=False, scale=8, lines=2)
        send = gr.Button('Отправить', variant='primary', scale=1)
    with gr.Row():
        clear = gr.Button('Очистить чат')
        export = gr.Button('Экспортировать историю JSON')
        download = gr.File(label='Файл истории', interactive=False)
    message.submit(respond, [message, state], [chatbot, message]).then(lambda h: h, [chatbot], [state])
    send.click(respond, [message, state], [chatbot, message]).then(lambda h: h, [chatbot], [state])
    clear.click(clear_chat, outputs=[chatbot, state, download])
    def export_history():
        path = Path('/content/kimi-history-export.json')
        path.write_text(json.dumps(load_saved_history(), ensure_ascii=False, indent=2), encoding='utf-8')
        return str(path)
    export.click(export_history, outputs=download)

print('Веб-чат запускается. Публичная ссылка будет временной и доступна, пока активна сессия.')
demo.queue().launch(share=True, debug=False)

## Если модель не загрузилась

В ячейке с `MODEL_ID` укажите `Qwen/Qwen2.5-3B-Instruct`, затем выполните ячейки повторно. Если возникла ошибка нехватки памяти, выберите GPU в настройках Colab, уменьшите `MAX_NEW_TOKENS` или используйте меньшую модель.

Публичная ссылка Gradio не является постоянным хостингом. Остановите сессию после использования. История сохраняется в Google Drive только при успешном подключении Drive.